Meet-the-Neighbors Colab notebook.

# Whole-genome VF prediction

This notebook runs the **genome-only** VF prediction on a small set of genomes **It is intended for Google Colab, where RAM, disk, and GPU time are limited.**

## Inputs

Upload one pair of files per genome:

* One `.gff` annotation file.
* One matching protein FASTA file ending in `.faa` or `.fasta`.

The files are paired by shared basename. These are valid examples:

* `GenomeA.gff` and `GenomeA.faa`
* `GenomeB.gff` and `GenomeB.fasta`

By default, this notebook accepts up to 3 genomes. Change `MAX_GENOMES` in the
setup cell if you have a larger Colab runtime and want to allow more.

## Output

Downloads the VF predictions `neighborhood_based_predictions.tsv` and visualizes the predictions along the genome.

In [ ]:
#@title 1. Install Meet-the-Neighbors, MMseqs2, Foldseek, and gLM weights


from pathlib import Path
import importlib.util
import os
import platform
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/mcn3159/meet-the-neighbors.git"
REPO_REF = "colab_run"

WORK_ROOT = Path("/content/meetneighbors_colab")
TOOLS_DIR = WORK_ROOT / "tools"
UPLOAD_DIR = WORK_ROOT / "uploaded_genomes"
RUN_DIR = WORK_ROOT / "run"
OUTPUT_DIR = RUN_DIR
FINAL_DIR = WORK_ROOT / "final_results"
GENOME_TSV = RUN_DIR / "genome_pairs.tsv"
FINAL_OUTPUT = FINAL_DIR / "neighborhood_based_predictions.tsv"

MAX_GENOMES = 3
threads = 2
mem_gb = 12
glm_batch_size = 50

WORK_ROOT.mkdir(parents=True, exist_ok=True)
TOOLS_DIR.mkdir(parents=True, exist_ok=True)

# Make tools visible to Python subprocesses in this notebook session.
os.environ["PATH"] = f"{TOOLS_DIR}:{os.environ.get('PATH', '')}"
!apt-get install -y aria2

def run_cmd(cmd, cwd=None):
    printable = " ".join(str(x) for x in cmd)
    print(f"$ cd {cwd} && {printable}" if cwd else f"$ {printable}")
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)


def has_cpu_flag(flag: str) -> bool:
    try:
        return flag in Path("/proc/cpuinfo").read_text()
    except Exception:
        return False


def download(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Already downloaded: {dest}")
        return

    if shutil.which("aria2c"):
        run_cmd([
            "aria2c",
            "-x", "16",
            "-s", "16",
            "-k", "1M",
            "--file-allocation=none",
            "-c",
            "-o", dest.name,
            "-d", str(dest.parent),
            url,
        ])
    else:
        run_cmd(["wget", "-q", "--show-progress", "-O", dest, url])


def install_mmseqs2_static():
    if shutil.which("mmseqs"):
        print("mmseqs already available:", shutil.which("mmseqs"))
        return

    if platform.machine() not in {"x86_64", "AMD64"}:
        raise RuntimeError("This Colab install cell currently expects x86_64 Linux.")

    if has_cpu_flag("avx2"):
        archive_name = "mmseqs-linux-avx2.tar.gz"
    elif has_cpu_flag("sse4_1"):
        archive_name = "mmseqs-linux-sse41.tar.gz"
    else:
        archive_name = "mmseqs-linux-sse2.tar.gz"

    archive = WORK_ROOT / archive_name
    download(f"https://mmseqs.com/latest/{archive_name}", archive)

    extract_dir = WORK_ROOT / "mmseqs"
    if not extract_dir.exists():
        run_cmd(["tar", "xzf", archive, "-C", WORK_ROOT])

    src = extract_dir / "bin" / "mmseqs"
    dst = TOOLS_DIR / "mmseqs"
    if not dst.exists():
        dst.symlink_to(src)

    print("mmseqs installed:", shutil.which("mmseqs"))


def install_foldseek_static():
    if shutil.which("foldseek"):
        print("foldseek already available:", shutil.which("foldseek"))
        return

    if platform.machine() not in {"x86_64", "AMD64"}:
        raise RuntimeError("This Colab install cell currently expects x86_64 Linux.")

    if not has_cpu_flag("avx2"):
        raise RuntimeError(
            "Foldseek's recommended Colab-style static binary is AVX2. "
            "This CPU does not report AVX2; use conda fallback for foldseek."
        )

    archive_name = "foldseek-linux-avx2.tar.gz"
    archive = WORK_ROOT / archive_name
    download(f"https://mmseqs.com/foldseek/{archive_name}", archive)

    extract_dir = WORK_ROOT / "foldseek"
    if not extract_dir.exists():
        run_cmd(["tar", "xzf", archive, "-C", WORK_ROOT])

    src = extract_dir / "bin" / "foldseek"
    dst = TOOLS_DIR / "foldseek"
    if not dst.exists():
        dst.symlink_to(src)

    print("foldseek installed:", shutil.which("foldseek"))


def install_python_package():
    # Direct install from GitHub branch/ref. Faster and cleaner than clone + checkout + pip install .
    pkg_url = f"git+{REPO_URL}@{REPO_REF}#egg=meetneighbors[colab]"
    run_cmd([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        # "--force-reinstall", # might cook versions
        pkg_url,
    ])

def download_html_templates():
  file_urls = [f"https://raw.githubusercontent.com/mcn3159/meet-the-neighbors/colab_run/notebooks/genome_cgview_template.html",
               f"https://raw.githubusercontent.com/mcn3159/meet-the-neighbors/colab_run/notebooks/vfsubcategory_cgview_genome_template.html"
  ]
  for file_url in file_urls:
    if shutil.which("aria2c"):
      run_cmd([
          "aria2c",
          "--file-allocation=none",
          "-c",
          file_url,
      ])
    else:
      run_cmd(["wget",file_url])

def get_package_root(package_name="meetneighbors") -> Path:
    spec = importlib.util.find_spec(package_name)
    if spec is None or spec.origin is None:
        raise RuntimeError(f"Could not find installed package: {package_name}")
    return Path(spec.origin).resolve().parent


def install_glm_model():
    package_root = get_package_root("meetneighbors")
    model_dir = package_root / "predictvfs" / "glm" / "model"
    model_dir.mkdir(parents=True, exist_ok=True)

    model_file = model_dir / "glm.bin"
    if model_file.exists() and model_file.stat().st_size > 0:
        print("gLM model already present:", model_file)
        return

    tmp_model = WORK_ROOT / "glm.bin"
    download("https://zenodo.org/records/7855545/files/glm.bin?download=1", tmp_model)

    shutil.copy2(tmp_model, model_file)
    print("gLM model installed:", model_file)


install_mmseqs2_static()
install_foldseek_static()
install_python_package()
download_html_templates()
install_glm_model()
os.environ["MPLBACKEND"] = "Agg" #matplotlib error fix


missing_tools = [
    tool for tool in ("meetneighbors", "mmseqs", "foldseek")
    if shutil.which(tool) is None
]
if missing_tools:
    raise RuntimeError(
        "Install finished, but these tools were not found on PATH: "
        + ", ".join(missing_tools)
    )

print("Install complete.")
print("meetneighbors:", shutil.which("meetneighbors"))
print("mmseqs:", shutil.which("mmseqs"))
print("foldseek:", shutil.which("foldseek"))

run_cmd(["mmseqs", "version"])
run_cmd(["foldseek", "version"])

## Upload genome files

Drag and drop all files together when prompted.

Each genome must have exactly one `.gff` and exactly one matching protein FASTA
file. Protein FASTA files may end in `.faa` or `.fasta`.

The notebook pairs files by basename, so `GenomeA.gff` pairs with
`GenomeA.faa`, and `GenomeB.gff` pairs with `GenomeB.fasta`.

In [ ]:
#@title 2. Upload `.gff` and `.faa`/`.fasta` genome pairs
#@markdown Click the upload button and select all genome files for this run.
#@markdown Upload no more than `3` `.gff` files and their matching
#@markdown protein FASTA files.

try:
    from google.colab import files
except ImportError as exc:
    raise RuntimeError(
        "This upload cell is intended for Google Colab, where "
        "`google.colab.files.upload()` is available."
    ) from exc

if UPLOAD_DIR.exists():
    shutil.rmtree(UPLOAD_DIR)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
if not uploaded:
    raise ValueError("No files were uploaded.")

uploaded_paths = []
for uploaded_name in uploaded:
    source = Path(uploaded_name)
    normalized_name = source.stem + source.suffix.lower()
    destination = UPLOAD_DIR / normalized_name
    if destination.exists():
        raise ValueError(
            f"Duplicate uploaded filename after extension normalization: "
            f"{destination.name}"
        )
    shutil.move(str(source), destination)
    uploaded_paths.append(destination)

print("Uploaded files:")
for path in uploaded_paths:
    print(f"  {path.name}")

#Validate genome pairs and build `genome_pairs.tsv`
#this creates the three-column TSV used by `meetneighbors --genome_tsv`.
#The columns are genome name, protein FASTA path, and GFF path.

PROTEIN_EXTENSIONS = {".faa", ".fasta"}
SUPPORTED_EXTENSIONS = PROTEIN_EXTENSIONS | {".gff"}


def uploaded_file_stem(path):
    suffix = path.suffix.lower()
    return path.name[: -len(suffix)]

# print(f"Detected {len(genome_pairs)} genome pair(s):")
# for genome_name, protein_path, gff_path in genome_pairs:
#     print(f"  {genome_name}: {gff_path.name} + {protein_path.name}")


## Run Meet-the-Neighbors

This step can take a while because it computes protein language model and
genomic language model embeddings. GPU runtimes are recommended when available.

The command uses `predictvf` in genome-only mode through `--genome_tsv`.

In [ ]:
#@title 3. Run Meet-the-Neighbors

import os
import shutil
import subprocess
from pathlib import Path

try:
    import torch
    gpu = 1 if torch.cuda.is_available() else 0
except Exception:
    gpu = 0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

predict_command = [
    "meetneighbors",
    "predictvf",
    "--genomes", str(UPLOAD_DIR),
    "--out", str(OUTPUT_DIR),
    "--threads", str(10),
    "--mem", str(mem_gb),
    "--glm_bs", str(glm_batch_size),
    "--gpu", str(gpu),
    "--memory_optimize",
    "--remove_temp",
    "--cluster",
    "-ns", "30000",
    "-ig", "10000",
]

# Only resume if the output directory already has content, not merely because it exists.
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    predict_command.append("--resume")

print("Running Meet-the-Neighbors with this command:")
print(" ".join(str(part) for part in predict_command))

process = subprocess.Popen(
    predict_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=os.environ.copy(),
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, predict_command)

## Download the final prediction table

This notebook intentionally keeps only one output file:

`neighborhood_based_predictions.tsv`

All other pipeline files are treated as intermediates.

In [ ]:
# #@title 6. Save, preview, and download `neighborhood_based_predictions.tsv`
# #@markdown The final TSV is copied into a small results directory before
# #@markdown cleanup starts.

# import pandas as pd

# pipeline_output = OUTPUT_DIR / "neighborhood_based_predictions.tsv"
# if not pipeline_output.exists():
#     raise FileNotFoundError(
#         f"Expected output was not found: {pipeline_output}. "
#         "Check the previous cell logs for the pipeline error."
#     )

# FINAL_DIR.mkdir(parents=True, exist_ok=True)
# shutil.copy2(pipeline_output, FINAL_OUTPUT)

# print(f"Final output saved to {FINAL_OUTPUT}")
# preview_df = pd.read_csv(FINAL_OUTPUT, sep="\t")
# print(f"Rows: {preview_df.shape[0]}, columns: {preview_df.shape[1]}")
# display(preview_df.head())

# files.download(str(FINAL_OUTPUT))

## 4. Visualize

This cell creates a link to visualize your VF predictions along a genome of interest.

In [ ]:
import subprocess, time, socket,urllib
from google.colab import output
import portpicker
import pandas as pd
import json
import base64
from IPython.display import HTML, display

# Define gff file and lc to VF mapping to the HTML
gff_filename = "NR2335.gff" #@param {type:"string"}
#@markdown (GenomeA.gff)
genome = gff_filename.split('.gff')[0]

# process and filter predictions for likely VFs
probability_threshold = 0.6
nn_preds = pd.read_csv(str(OUTPUT_DIR) + '/neighborhood_based_predictions.tsv',sep="\t")
print(nn_preds.shape)
nn_preds.drop_duplicates(subset='query',inplace=True)
print("NN preds shape after remove dups:",nn_preds.shape)
nn_preds['genome'] = nn_preds['query'].str.split('_').str[0]
print("Genomes in preds:",set(nn_preds['genome']))
# genomes = [str(genome).split('/')[-1] for genome in UPLOAD_DIR.iterdir() if '.gff' in str(genome)]
nn_preds = nn_preds[nn_preds['genome'] == genome]
og_shape = nn_preds.shape
print(og_shape)
mask = (nn_preds.iloc[:,2:-1] >= probability_threshold).any(axis=1)
nn_preds = nn_preds.loc[mask]
print(f"{og_shape[0] - nn_preds.shape[0]} rows have 'iffy' predictions where no cat had a probability >= {probability_threshold}")
nn_preds['prediction'] = nn_preds.iloc[:,2:-1].idxmax(axis=1)
nn_preds = nn_preds[nn_preds['prediction']!='non_vf'] # save mapping of locus tag to VF predictions
lc_pred = dict(zip(nn_preds['query'],nn_preds['prediction']))
with open('/content/lc_vfprediction.json','w') as f:
  json.dump(lc_pred,f,indent=2)


gff_path = Path(UPLOAD_DIR) / gff_filename # lc_pred is your Python dictionary: # {"NR2335_003645": "Type II secretion system (T2SS)", ...}
with open("/content/genome_cgview_template.html") as f:
  html = f.read()
gff_text_b64 = base64.b64encode(gff_path.read_bytes()).decode("ascii")
category_json = json.dumps(lc_pred)
html = html.replace("__GFF_FILENAME__", gff_filename)
html = html.replace("__GFF_TEXT_B64__", gff_text_b64)
html = html.replace("__CATEGORY_BY_LOCUS_TAG_JSON__", category_json)
with open("/content/index.html", "w") as f:
    f.write(html)

import html as html_lib

with open("/content/index.html", "r", encoding="utf-8") as f:
    html_text = f.read()

display(HTML(f"""
<iframe
  srcdoc="{html_lib.escape(html_text, quote=True)}"
  width="950"
  height="950"
  style="border:1px solid #ccc;">
</iframe>
"""))

In [ ]:
# @title 5. Visualize gLM-based predictions + MMseqs2 for addtional detail in VF annotations
# @markdown Runs at the cost of sensitivity
import requests
from Bio import SeqIO

# -----------------------------
# Settings
# -----------------------------
repo = "mcn3159/expanded_vfdb"
db_subdir = "db"

work_dir = Path("/content/expanded_vfdb_mmseqs")
download_dir = work_dir / "expanded_vfdb_db"
tmp_dir = work_dir / "tmp"

query_fasta = work_dir / "nn_preds_query_proteins.faa"
target_fasta = work_dir / "expanded_vfdb_target.faa"

query_db = work_dir / "querydb"
search_res = work_dir / "searchres"
out_tsv = work_dir / "expanded_vfdb_mmseqs_hits.tsv"

work_dir.mkdir(parents=True, exist_ok=True)
download_dir.mkdir(parents=True, exist_ok=True)
tmp_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Small helper
# -----------------------------
def run(cmd):
    print(" ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

# -----------------------------
# Download GitHub db files,
# excluding files with _ca or _ss in the filename
# -----------------------------
repo_info = requests.get(f"https://api.github.com/repos/{repo}").json()
branch = repo_info["default_branch"]

tree = requests.get(
    f"https://api.github.com/repos/{repo}/git/trees/{branch}?recursive=1"
).json()["tree"]

db_files = [
    item["path"]
    for item in tree
    if item["type"] == "blob"
    # and item["path"].startswith(f"{db_subdir}/")
    and "_ca" not in Path(item["path"]).name
    and "_ss" not in Path(item["path"]).name
]

print(f"Downloading {len(db_files)} files from {repo}/{db_subdir}")

downloaded = []

for rel_path in db_files:
    url = f"https://raw.githubusercontent.com/{repo}/{branch}/{rel_path}"
    local_path = download_dir / Path(rel_path).name

    r = requests.get(url)
    r.raise_for_status()

    local_path.write_bytes(r.content)
    downloaded.append(local_path)
target_db = local_path.parent / "expanded_vfdb_foldseekdb"
# -----------------------------
# Keep only allowed FASTA suffixes
# -----------------------------
allowed_fasta_suffixes = {
    ".faa",
    ".fa",
    ".fasta",
    ".fas",
}

# -----------------------------
# Find the user's protein FASTA corresponding to the GFF
# -----------------------------
genome = Path(gff_filename).name.replace(".gff", "")

search_roots = [Path(UPLOAD_DIR), Path("/content")]
protein_fasta_candidates = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        suffix = "".join(path.suffixes[-2:]) if path.name.endswith(".gz") else path.suffix

        if path.name.startswith(genome) and suffix in allowed_fasta_suffixes:
            protein_fasta_candidates.append(path)

if not protein_fasta_candidates:
    raise FileNotFoundError(
        f"Could not find protein FASTA for {gff_filename}. "
        f"Expected something like {genome}.faa, {genome}.fa, or {genome}.fasta."
    )

protein_fasta = sorted(
    protein_fasta_candidates,
    key=lambda p: (
        0 if p.name.endswith(".faa") or p.name.endswith(".faa.gz") else
        1 if p.name.endswith(".fasta") or p.name.endswith(".fasta.gz") else
        2 if p.name.endswith(".fa") or p.name.endswith(".fa.gz") else
        3
    )
)[0]

print("Using query protein FASTA:", protein_fasta)

# -----------------------------
# Write query FASTA from nn_preds['query']
# -----------------------------
query_id_set = set(nn_preds.dropna(subset='query')["query"])
seq_by_id = {rec.id:rec for rec in SeqIO.parse(protein_fasta,'fasta') if (
    rec.id.split('|')[-1] in query_id_set) or (('|' in rec.id) and (rec.id.split('|')[1] in query_id_set))}

print(f"Matched {len(seq_by_id)} of {len(query_id_set)} nn_preds query proteins")

SeqIO.write(list(seq_by_id.values()),open(query_fasta,"w"),"fasta")

# -----------------------------
# Clean previous MMseqs outputs from reruns
# -----------------------------
for prefix in [query_db, target_db, search_res]:
    for path in work_dir.glob(prefix.name + "*"):
        if path.is_file():
            path.unlink()
        elif path.is_dir():
            shutil.rmtree(path)

if tmp_dir.exists():
    shutil.rmtree(tmp_dir)
tmp_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Build MMseqs databases
# -----------------------------
run(["mmseqs", "createdb", query_fasta, query_db])

# -----------------------------
# Run MMseqs search
# -----------------------------
run([
    "mmseqs", "search",
    query_db,
    target_db,
    search_res,
    tmp_dir,
    "-s", "7.5",
    "-c", "0.8",
    "--cov-mode", "0",
    "--alignment-mode", "3",
    "--max-seqs", "10000",
    "--threads", str(threads),
    "-a",
    "--min-seq-id", "0.3",
])

# -----------------------------
# Convert alignments to TSV
# -----------------------------
res_headers = "query,target,qheader,theader,evalue,pident,qcov,fident,alnlen,bits"
run([
    "mmseqs", "convertalis",
    query_db,
    target_db,
    search_res,
    out_tsv,
    "--threads", str(threads),
    "--format-output",
    res_headers,
])

# # -----------------------------
# # Read results into pandas
# # -----------------------------
vfdb_mmseqs_hits = pd.read_csv(
    out_tsv,
    sep="\t",
    names=res_headers.split(",")
)

print("MMseqs hits:", vfdb_mmseqs_hits.shape)
vfdb_mmseqs_hits.head()

# read in cat labels
cat_labels = pd.read_csv(local_path.parent / "category_assignments_all.tsv",sep="\t")
vfdb_mmseqs_hits = pd.merge(vfdb_mmseqs_hits,cat_labels[["target","vf_category","VF_subcategory_high","VF_subcategory_low"]].drop_duplicates()
         ,how="left",on="target")
vfdb_mmseqs_hits.dropna(subset='vf_category',inplace=True) #FIND MISSING VFS LATER
vfdb_mmseqs_hits = vfdb_mmseqs_hits.sort_values(by='bits',ascending=False).drop_duplicates(subset='query',keep="first")
vf_label_json = vfdb_mmseqs_hits.set_index("query")[["vf_category", "VF_subcategory_low"]].to_dict(orient='index')
with open("/content/vfsubcategory_cgview_genome_template.html") as f:
    html = f.read()

html = html.replace("__GFF_FILENAME__", gff_filename)
html = html.replace("__GFF_TEXT_B64__", base64.b64encode(gff_path.read_bytes()).decode("ascii"))
html = html.replace("__CATEGORY_BY_LOCUS_TAG_JSON__", json.dumps(vf_label_json))

with open("/content/index.html", "w") as f:
    f.write(html)

import html as html_lib

with open("/content/index.html", "r", encoding="utf-8") as f:
    html_text = f.read()

display(HTML(f"""
<iframe
  srcdoc="{html_lib.escape(html_text, quote=True)}"
  width="950"
  height="950"
  style="border:1px solid #ccc;">
</iframe>
"""))